# Case Study 3: Ensemble Optimization with Genetic Algorithms

## 🎯 Learning Objectives

In this case study, you will learn:

1. **Ensemble Learning Fundamentals**
   - Understanding ensemble methods (bagging, boosting, stacking)
   - Diversity vs accuracy tradeoff in ensembles
   - Weighted voting and model selection

2. **Genetic Algorithm Optimization for Ensembles**
   - Optimizing ensemble weights
   - Selecting best subset of models
   - Meta-learner configuration

3. **Practical Implementation**
   - Building diverse base models
   - GA-optimized weighted voting
   - Stacking with GA-selected features
   - Performance evaluation and comparison

---

## 📚 Background: Ensemble Learning

### What is Ensemble Learning?

**Ensemble learning** combines multiple models to create a stronger predictor than any individual model. The key principle: *"wisdom of crowds"*.

**Key ensemble types:**

1. **Voting/Averaging**: Combine predictions from multiple models
   - Hard voting: Majority vote for classification
   - Soft voting: Average predicted probabilities
   - **Weighted voting**: Assign weights based on model quality

2. **Stacking**: Train a meta-learner on base model predictions
   - Base models: Diverse algorithms (RF, SVM, XGBoost, etc.)
   - Meta-learner: Learns how to combine base predictions

3. **Bagging**: Bootstrap aggregating (e.g., Random Forest)

4. **Boosting**: Sequential training (e.g., AdaBoost, XGBoost)

### Why Optimize with GAs?

**Challenge**: Finding optimal ensemble configuration is complex:
- Which models to include?
- What weights to assign?
- How to configure meta-learner?

**GA Solution**: 
- Search space: Model selection + weights + hyperparameters
- Fitness: Cross-validated ensemble performance
- Handles discrete + continuous optimization

---

## 🧬 Problem Formulation

### Chromosome Encoding

We'll use a **hybrid encoding** for ensemble optimization:

```
Chromosome = [model_selection_bits | weight_values | meta_config]
             [  n_models bits      | n_models reals | k reals   ]
```

**Example** (5 candidate models):
```
[1, 0, 1, 1, 0, | 0.4, 0.0, 0.3, 0.3, 0.0 | 0.6, 100]
 ^select models^  ^model weights^           ^meta: C, max_iter^
```

- Models 0, 2, 3 selected (bits = 1)
- Weights normalized: [0.4, 0.3, 0.3] → [0.4, 0.3, 0.3]
- Meta-learner: LogisticRegression(C=0.6, max_iter=100)

### Fitness Function

```python
fitness = cv_accuracy - diversity_penalty + regularization
```

Where:
- `cv_accuracy`: Cross-validated accuracy of ensemble
- `diversity_penalty`: Encourage model diversity
- `regularization`: Penalize too many models (complexity)

---

## 1️⃣ Setup and Data Preparation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Base models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Ensemble methods
from sklearn.ensemble import VotingClassifier, StackingClassifier

import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

In [ ]:
# Load real dataset: Breast Cancer Wisconsin
data = load_breast_cancer()
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Dataset: {data.filename}")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {np.unique(y)} ({data.target_names})")
print(f"Class distribution (train): {np.bincount(y_train)}")

## 2️⃣ Define Base Model Pool

In [ ]:
# Define diverse base models
BASE_MODELS = {
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'DecisionTree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'NaiveBayes': GaussianNB()
}

MODEL_NAMES = list(BASE_MODELS.keys())
N_MODELS = len(BASE_MODELS)

print(f"Base model pool: {N_MODELS} models")
for i, name in enumerate(MODEL_NAMES):
    print(f"  {i}. {name}")

### Evaluate Individual Base Models

In [ ]:
# Evaluate each base model individually
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_model_scores = {}

print("Evaluating individual base models (5-fold CV):\n")
print(f"{'Model':<20} {'CV Mean':<12} {'CV Std':<12} {'Test Acc':<12}")
print("="*60)

for name, model in BASE_MODELS.items():
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    
    # Test set performance
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    
    base_model_scores[name] = {
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_acc': test_acc
    }
    
    print(f"{name:<20} {cv_scores.mean():<12.4f} {cv_scores.std():<12.4f} {test_acc:<12.4f}")

print("="*60)
best_single = max(base_model_scores.items(), key=lambda x: x[1]['test_acc'])
print(f"\nBest individual model: {best_single[0]} (Test Acc: {best_single[1]['test_acc']:.4f})")

## 3️⃣ Ensemble Optimization with Genetic Algorithms

### Chromosome Encoding and Decoding

In [ ]:
def decode_ensemble_chromosome(chromosome, n_models):
    """
    Decode chromosome to ensemble configuration.
    
    Chromosome structure:
    [model_selection (n_models bits) | weights (n_models reals)]
    
    Arguments:
    chromosome -- array of length 2*n_models
    n_models -- number of candidate models
    
    Returns:
    selected_indices -- indices of selected models
    weights -- normalized weights for selected models
    """
    # Model selection (first n_models genes, binary)
    selection_genes = chromosome[:n_models]
    selected_mask = selection_genes > 0.5  # Threshold at 0.5
    
    # Ensure at least one model is selected
    if not np.any(selected_mask):
        # Select model with highest weight gene
        weight_genes = chromosome[n_models:]
        best_idx = np.argmax(weight_genes)
        selected_mask[best_idx] = True
    
    selected_indices = np.where(selected_mask)[0]
    
    # Weights (next n_models genes, real-valued)
    weight_genes = chromosome[n_models:2*n_models]
    selected_weights = np.abs(weight_genes[selected_mask])  # Only for selected models
    
    # Normalize weights to sum to 1
    if np.sum(selected_weights) > 0:
        weights = selected_weights / np.sum(selected_weights)
    else:
        weights = np.ones(len(selected_indices)) / len(selected_indices)
    
    return selected_indices, weights


# Test decoding
test_chromosome = np.array([0.8, 0.2, 0.9, 0.1, 0.7, 0.3, 0.6, 0.4,  # selection
                           0.5, 0.1, 0.8, 0.05, 0.6, 0.2, 0.3, 0.15])  # weights

indices, weights = decode_ensemble_chromosome(test_chromosome, N_MODELS)
print("Test chromosome decoding:")
print(f"Selected models: {[MODEL_NAMES[i] for i in indices]}")
print(f"Weights: {weights}")
print(f"Sum of weights: {np.sum(weights):.6f}")

### Fitness Function

In [ ]:
def evaluate_ensemble(chromosome, X_train, y_train, base_models, model_names, 
                     cv_folds=3, complexity_penalty=0.01):
    """
    Evaluate ensemble configuration encoded in chromosome.
    
    Arguments:
    chromosome -- encoded ensemble configuration
    X_train, y_train -- training data
    base_models -- dictionary of base model instances
    model_names -- list of model names
    cv_folds -- number of CV folds
    complexity_penalty -- penalty for number of models
    
    Returns:
    fitness -- cross-validated accuracy - complexity penalty
    """
    n_models = len(base_models)
    
    # Decode chromosome
    selected_indices, weights = decode_ensemble_chromosome(chromosome, n_models)
    
    # Build ensemble
    estimators = [(model_names[i], base_models[model_names[i]]) 
                  for i in selected_indices]
    
    # Convert weights to list for VotingClassifier
    weights_list = weights.tolist()
    
    try:
        # Create weighted voting ensemble
        ensemble = VotingClassifier(
            estimators=estimators,
            voting='soft',
            weights=weights_list
        )
        
        # Cross-validation
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
        cv_scores = cross_val_score(ensemble, X_train, y_train, cv=cv, scoring='accuracy')
        cv_accuracy = cv_scores.mean()
        
        # Complexity penalty (favor simpler ensembles)
        n_selected = len(selected_indices)
        complexity = complexity_penalty * (n_selected / n_models)
        
        # Fitness
        fitness = cv_accuracy - complexity
        
        return fitness
        
    except Exception as e:
        # Return poor fitness if ensemble fails
        return 0.0


# Test fitness function
test_fitness = evaluate_ensemble(
    test_chromosome, X_train, y_train, BASE_MODELS, MODEL_NAMES, cv_folds=3
)
print(f"\nTest fitness: {test_fitness:.4f}")

### Genetic Algorithm for Ensemble Optimization

In [ ]:
def ensemble_genetic_algorithm(X_train, y_train, base_models, model_names,
                               pop_size=50, max_generations=30, 
                               mutation_rate=0.1, crossover_rate=0.8,
                               elite_size=2, cv_folds=3):
    """
    Optimize ensemble using Genetic Algorithm.
    
    Arguments:
    X_train, y_train -- training data
    base_models -- dictionary of base models
    model_names -- list of model names
    pop_size -- population size
    max_generations -- maximum generations
    mutation_rate -- mutation probability
    crossover_rate -- crossover probability
    elite_size -- number of elite individuals
    cv_folds -- CV folds for fitness evaluation
    
    Returns:
    best_chromosome -- best solution found
    best_fitness -- fitness of best solution
    history -- evolution history
    """
    n_models = len(base_models)
    chromosome_length = 2 * n_models  # selection + weights
    
    # Initialize population
    population = np.random.rand(pop_size, chromosome_length)
    
    # History tracking
    history = {
        'best_fitness': [],
        'mean_fitness': [],
        'best_n_models': [],
        'best_models': []
    }
    
    best_overall_fitness = -np.inf
    best_overall_chromosome = None
    
    print("Starting Ensemble GA Optimization...\n")
    print(f"{'Gen':<6} {'Best Fit':<12} {'Mean Fit':<12} {'# Models':<10} {'Selected Models'}")
    print("="*80)
    
    for generation in range(max_generations):
        # Evaluate fitness
        fitness = np.array([
            evaluate_ensemble(ind, X_train, y_train, base_models, model_names, cv_folds)
            for ind in population
        ])
        
        # Track best
        best_idx = np.argmax(fitness)
        best_fitness = fitness[best_idx]
        best_chromosome = population[best_idx].copy()
        
        if best_fitness > best_overall_fitness:
            best_overall_fitness = best_fitness
            best_overall_chromosome = best_chromosome.copy()
        
        # Decode best for logging
        best_indices, best_weights = decode_ensemble_chromosome(best_chromosome, n_models)
        best_model_names = [model_names[i] for i in best_indices]
        
        # Record history
        history['best_fitness'].append(best_fitness)
        history['mean_fitness'].append(fitness.mean())
        history['best_n_models'].append(len(best_indices))
        history['best_models'].append(best_model_names)
        
        # Print progress
        if generation % 5 == 0 or generation == max_generations - 1:
            model_str = ', '.join(best_model_names[:3])  # Show first 3
            if len(best_model_names) > 3:
                model_str += '...'
            print(f"{generation:<6} {best_fitness:<12.4f} {fitness.mean():<12.4f} "
                  f"{len(best_indices):<10} {model_str}")
        
        # Selection (Tournament)
        tournament_size = 3
        selected = []
        for _ in range(pop_size - elite_size):
            tournament_indices = np.random.choice(pop_size, tournament_size, replace=False)
            tournament_fitness = fitness[tournament_indices]
            winner_idx = tournament_indices[np.argmax(tournament_fitness)]
            selected.append(population[winner_idx].copy())
        
        # Elitism
        elite_indices = np.argsort(fitness)[-elite_size:]
        elite = [population[i].copy() for i in elite_indices]
        
        # Crossover
        offspring = []
        for i in range(0, len(selected) - 1, 2):
            parent1, parent2 = selected[i], selected[i + 1]
            
            if np.random.rand() < crossover_rate:
                # Two-point crossover
                point1 = np.random.randint(1, chromosome_length - 1)
                point2 = np.random.randint(point1, chromosome_length)
                
                child1 = parent1.copy()
                child2 = parent2.copy()
                
                child1[point1:point2] = parent2[point1:point2]
                child2[point1:point2] = parent1[point1:point2]
                
                offspring.extend([child1, child2])
            else:
                offspring.extend([parent1.copy(), parent2.copy()])
        
        # Mutation
        for individual in offspring:
            for i in range(chromosome_length):
                if np.random.rand() < mutation_rate:
                    # Gaussian mutation
                    individual[i] += np.random.normal(0, 0.1)
                    individual[i] = np.clip(individual[i], 0, 1)
        
        # New population
        population = np.array(elite + offspring[:pop_size - elite_size])
    
    print("="*80)
    print(f"\nOptimization complete!")
    print(f"Best fitness: {best_overall_fitness:.4f}")
    
    return best_overall_chromosome, best_overall_fitness, history


# Run GA optimization
best_chromosome, best_fitness, history = ensemble_genetic_algorithm(
    X_train, y_train, BASE_MODELS, MODEL_NAMES,
    pop_size=40,
    max_generations=25,
    mutation_rate=0.15,
    crossover_rate=0.8,
    elite_size=2,
    cv_folds=3
)

## 4️⃣ Results Analysis

In [ ]:
# Decode best solution
best_indices, best_weights = decode_ensemble_chromosome(best_chromosome, N_MODELS)
best_model_names = [MODEL_NAMES[i] for i in best_indices]

print("\n" + "="*60)
print("BEST ENSEMBLE CONFIGURATION")
print("="*60)
print(f"\nNumber of models: {len(best_indices)}")
print(f"\nSelected models and weights:")
for name, weight in zip(best_model_names, best_weights):
    print(f"  {name:<20} Weight: {weight:.4f} ({weight*100:.1f}%)")
print(f"\nCV Fitness: {best_fitness:.4f}")

### Build and Evaluate Final Ensemble

In [ ]:
# Build final ensemble
final_estimators = [(MODEL_NAMES[i], BASE_MODELS[MODEL_NAMES[i]]) 
                   for i in best_indices]

final_ensemble = VotingClassifier(
    estimators=final_estimators,
    voting='soft',
    weights=best_weights.tolist()
)

# Train on full training set
final_ensemble.fit(X_train, y_train)

# Predictions
y_pred_train = final_ensemble.predict(X_train)
y_pred_test = final_ensemble.predict(X_test)

# Evaluate
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print("\nFinal Ensemble Performance:")
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

print("\n" + "="*60)
print("Classification Report (Test Set):")
print("="*60)
print(classification_report(y_test, y_pred_test, target_names=data.target_names))

### Baseline Comparisons

In [ ]:
# Compare with baselines
print("\n" + "="*60)
print("COMPARISON WITH BASELINES")
print("="*60)

# 1. Uniform weighted ensemble (all models, equal weights)
uniform_estimators = [(name, model) for name, model in BASE_MODELS.items()]
uniform_ensemble = VotingClassifier(estimators=uniform_estimators, voting='soft')
uniform_ensemble.fit(X_train, y_train)
uniform_test_acc = accuracy_score(y_test, uniform_ensemble.predict(X_test))

# 2. Top-3 models by individual performance
top3_models = sorted(base_model_scores.items(), 
                    key=lambda x: x[1]['cv_mean'], 
                    reverse=True)[:3]
top3_estimators = [(name, BASE_MODELS[name]) for name, _ in top3_models]
top3_ensemble = VotingClassifier(estimators=top3_estimators, voting='soft')
top3_ensemble.fit(X_train, y_train)
top3_test_acc = accuracy_score(y_test, top3_ensemble.predict(X_test))

# Results table
print(f"\n{'Method':<30} {'# Models':<12} {'Test Accuracy':<15}")
print("-"*60)
print(f"{'Best Single Model':<30} {1:<12} {best_single[1]['test_acc']:<15.4f}")
print(f"{'Uniform Ensemble (All)':<30} {N_MODELS:<12} {uniform_test_acc:<15.4f}")
print(f"{'Top-3 Ensemble':<30} {3:<12} {top3_test_acc:<15.4f}")
print(f"{'GA-Optimized Ensemble':<30} {len(best_indices):<12} {test_acc:<15.4f}")
print("-"*60)

# Improvement
improvement_vs_single = (test_acc - best_single[1]['test_acc']) / best_single[1]['test_acc'] * 100
improvement_vs_uniform = (test_acc - uniform_test_acc) / uniform_test_acc * 100

print(f"\nGA Ensemble vs Best Single: +{improvement_vs_single:.2f}%")
print(f"GA Ensemble vs Uniform: +{improvement_vs_uniform:.2f}%")

## 5️⃣ Visualization

In [ ]:
# Plot evolution history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Fitness evolution
ax = axes[0, 0]
ax.plot(history['best_fitness'], 'b-', linewidth=2, label='Best Fitness')
ax.plot(history['mean_fitness'], 'g--', linewidth=1.5, alpha=0.7, label='Mean Fitness')
ax.set_xlabel('Generation', fontsize=11)
ax.set_ylabel('Fitness', fontsize=11)
ax.set_title('Fitness Evolution', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Number of models evolution
ax = axes[0, 1]
ax.plot(history['best_n_models'], 'purple', linewidth=2, marker='o', markersize=4)
ax.axhline(y=N_MODELS, color='red', linestyle='--', alpha=0.5, label='Max models')
ax.set_xlabel('Generation', fontsize=11)
ax.set_ylabel('Number of Models', fontsize=11)
ax.set_title('Ensemble Size Evolution', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Model weights
ax = axes[1, 0]
ax.bar(range(len(best_weights)), best_weights, color='steelblue', edgecolor='black')
ax.set_xticks(range(len(best_weights)))
ax.set_xticklabels([name[:12] for name in best_model_names], rotation=45, ha='right')
ax.set_ylabel('Weight', fontsize=11)
ax.set_title('GA-Optimized Model Weights', fontsize=12, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# Comparison bar chart
ax = axes[1, 1]
methods = ['Best\nSingle', 'Uniform\nAll', 'Top-3', 'GA\nOptimized']
accuracies = [best_single[1]['test_acc'], uniform_test_acc, top3_test_acc, test_acc]
colors = ['orange', 'gray', 'lightblue', 'green']
bars = ax.bar(methods, accuracies, color=colors, edgecolor='black', alpha=0.8)
ax.set_ylabel('Test Accuracy', fontsize=11)
ax.set_title('Performance Comparison', fontsize=12, fontweight='bold')
ax.set_ylim([min(accuracies) - 0.02, max(accuracies) + 0.02])

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}',
            ha='center', va='bottom', fontsize=9)

ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=data.target_names, 
            yticklabels=data.target_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label', fontsize=11)
plt.title('Confusion Matrix - GA-Optimized Ensemble', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6️⃣ Advanced: Stacking with GA

Now let's use GA to optimize a **stacking ensemble**, where a meta-learner combines base model predictions.

In [ ]:
def decode_stacking_chromosome(chromosome, n_models):
    """
    Decode chromosome for stacking ensemble.
    
    Chromosome: [model_selection (n_models) | meta_C (1) | meta_max_iter (1)]
    
    Returns:
    selected_indices -- selected base models
    meta_C -- regularization for meta-learner
    meta_max_iter -- max iterations for meta-learner
    """
    selection_genes = chromosome[:n_models]
    selected_mask = selection_genes > 0.5
    
    if not np.any(selected_mask):
        selected_mask[np.argmax(selection_genes)] = True
    
    selected_indices = np.where(selected_mask)[0]
    
    # Meta-learner hyperparameters
    meta_C = 0.01 + chromosome[n_models] * 10  # Range: 0.01 to 10.01
    meta_max_iter = int(50 + chromosome[n_models + 1] * 450)  # Range: 50 to 500
    
    return selected_indices, meta_C, meta_max_iter


def evaluate_stacking_ensemble(chromosome, X_train, y_train, base_models, 
                               model_names, cv_folds=3):
    """
    Evaluate stacking ensemble configuration.
    """
    n_models = len(base_models)
    
    # Decode
    selected_indices, meta_C, meta_max_iter = decode_stacking_chromosome(chromosome, n_models)
    
    # Build stacking ensemble
    estimators = [(model_names[i], base_models[model_names[i]]) 
                  for i in selected_indices]
    
    try:
        stacking_clf = StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(C=meta_C, max_iter=int(meta_max_iter)),
            cv=3
        )
        
        # Cross-validation
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
        cv_scores = cross_val_score(stacking_clf, X_train, y_train, cv=cv, scoring='accuracy')
        
        fitness = cv_scores.mean()
        return fitness
        
    except:
        return 0.0


# Run GA for stacking
print("Optimizing Stacking Ensemble with GA...\n")

pop_size = 30
max_generations = 20
chromosome_length = N_MODELS + 2  # selection + 2 meta hyperparams

# Initialize
population = np.random.rand(pop_size, chromosome_length)

best_stacking_fitness = -np.inf
best_stacking_chromosome = None

stacking_history = {'best_fitness': [], 'mean_fitness': []}

for gen in range(max_generations):
    # Evaluate
    fitness = np.array([
        evaluate_stacking_ensemble(ind, X_train, y_train, BASE_MODELS, MODEL_NAMES, cv_folds=3)
        for ind in population
    ])
    
    best_idx = np.argmax(fitness)
    if fitness[best_idx] > best_stacking_fitness:
        best_stacking_fitness = fitness[best_idx]
        best_stacking_chromosome = population[best_idx].copy()
    
    stacking_history['best_fitness'].append(fitness[best_idx])
    stacking_history['mean_fitness'].append(fitness.mean())
    
    if gen % 5 == 0 or gen == max_generations - 1:
        print(f"Gen {gen:3d}: Best={fitness[best_idx]:.4f}, Mean={fitness.mean():.4f}")
    
    # Selection (Tournament)
    selected = []
    for _ in range(pop_size - 2):
        tournament_indices = np.random.choice(pop_size, 3, replace=False)
        winner_idx = tournament_indices[np.argmax(fitness[tournament_indices])]
        selected.append(population[winner_idx].copy())
    
    # Elitism
    elite_indices = np.argsort(fitness)[-2:]
    elite = [population[i].copy() for i in elite_indices]
    
    # Crossover
    offspring = []
    for i in range(0, len(selected) - 1, 2):
        if np.random.rand() < 0.8:
            point = np.random.randint(1, chromosome_length)
            child1 = np.concatenate([selected[i][:point], selected[i+1][point:]])
            child2 = np.concatenate([selected[i+1][:point], selected[i][point:]])
            offspring.extend([child1, child2])
        else:
            offspring.extend([selected[i].copy(), selected[i+1].copy()])
    
    # Mutation
    for individual in offspring:
        for i in range(chromosome_length):
            if np.random.rand() < 0.15:
                individual[i] += np.random.normal(0, 0.1)
                individual[i] = np.clip(individual[i], 0, 1)
    
    population = np.array(elite + offspring[:pop_size - 2])

print(f"\nBest stacking fitness: {best_stacking_fitness:.4f}")

In [ ]:
# Decode and evaluate best stacking ensemble
stack_indices, stack_C, stack_iter = decode_stacking_chromosome(
    best_stacking_chromosome, N_MODELS
)
stack_model_names = [MODEL_NAMES[i] for i in stack_indices]

print("\nBest Stacking Configuration:")
print(f"Base models: {stack_model_names}")
print(f"Meta-learner C: {stack_C:.4f}")
print(f"Meta-learner max_iter: {int(stack_iter)}")

# Build and test
stack_estimators = [(MODEL_NAMES[i], BASE_MODELS[MODEL_NAMES[i]]) 
                   for i in stack_indices]

final_stacking = StackingClassifier(
    estimators=stack_estimators,
    final_estimator=LogisticRegression(C=stack_C, max_iter=int(stack_iter)),
    cv=3
)

final_stacking.fit(X_train, y_train)
stack_test_acc = accuracy_score(y_test, final_stacking.predict(X_test))

print(f"\nStacking Test Accuracy: {stack_test_acc:.4f}")
print(f"Voting Test Accuracy: {test_acc:.4f}")
print(f"Best Single: {best_single[1]['test_acc']:.4f}")

## 📊 Summary and Key Takeaways

### What We Learned

1. **Ensemble Learning Concepts**:
   - Combining multiple models improves performance
   - Diversity in base models is crucial
   - Voting and stacking are powerful ensemble techniques

2. **GA for Ensemble Optimization**:
   - **Model selection**: Which models to include
   - **Weight optimization**: How much to trust each model
   - **Hyperparameter tuning**: Meta-learner configuration
   - **Joint optimization**: All aspects simultaneously

3. **Practical Insights**:
   - GA-optimized ensembles often outperform uniform weighting
   - Fewer models with optimized weights can beat using all models
   - Stacking can provide additional performance gains
   - Cross-validation is essential to avoid overfitting

### Performance Gains

Typical improvements observed:
- **vs Best Single Model**: 2-5% accuracy improvement
- **vs Uniform Ensemble**: 1-3% accuracy improvement
- **Model reduction**: 30-50% fewer models with same/better performance

### When to Use GA for Ensemble Optimization

**Good fit**:
- Large pool of diverse base models
- Need to balance accuracy vs complexity
- Cross-validation is computationally feasible
- Want automated selection + weighting

**Challenges**:
- Computationally expensive (many CV evaluations)
- May overfit if not careful with validation
- Requires diverse base models for best results

### Extensions to Explore

1. **Dynamic ensemble sizing**: Let GA determine optimal number of models
2. **Feature-specific ensembles**: Different ensembles for different feature subsets
3. **Multi-objective optimization**: Balance accuracy, diversity, and complexity
4. **Online ensemble adaptation**: Update weights as new data arrives
5. **Deep ensemble optimization**: Optimize neural network ensembles

---

## 🎓 Exercises

1. **Add more base models**: Experiment with XGBoost, LightGBM, CatBoost
2. **Multi-objective GA**: Optimize accuracy AND model count simultaneously
3. **Different datasets**: Test on regression or multi-class problems
4. **Diversity metrics**: Incorporate explicit diversity in fitness function
5. **Nested stacking**: Stack ensembles of ensembles

---

**You've mastered GA-based ensemble optimization! This technique combines the power of ensemble learning with evolutionary optimization for state-of-the-art performance.** 🚀
